# Step 51 — Composition counts in selected reporting windows

All step-specific code is below. It reads the trusted local catalogue pickle created by step 50 and replaces a daily CSV. It trusts the seven `event_class:` labels, including P_H, rather than inferring scientific content from temporary export headers. Unknown labels and records with no usable representative time are skipped and counted in the console summary.

## Window meaning and time choice

Default reporting windows are May 8–June 9 and June 30–August 18, 1991, inclusive through the end of each final day. These are **selected historical reporting windows**, not demonstrated complete earthquake detection or complete pick coverage. The notebook's legacy filename says “complete”; do not repeat that claim without an independently justified availability/completeness analysis. `windows=None` selects those defaults; supply explicit date pairs to change them.

Representative time priority: preferred origin, first timed origin, waveform-start comment, then earliest timed pick. Therefore this daily count may differ from step-33 waveform-start counts or step-53 S-file-filename dates. UTC representation does not establish the source clocks' absolute accuracy. Source-only classes are legitimate archive records and are retained.

Class columns partition the selected catalogue. An all-record catalogue and its waveform subset overlap; do not add their totals. Inspect the summary for skipped records and empty selected windows. Rebuilding the CSV never changes the catalogue. The implementation writes observed-day summaries; absence of a row is not an independently verified day with no earthquakes.

## Configuration

Inspect all input/output paths and settings. This cell resolves configuration; definitions and run cells follow.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

SETTINGS = dict(
    catalog_pkl=P['CATALOG_PREFIX'] + '.pkl',
    out_daily_csv=str(Path(P['TMP_DIR']) / '51_daily_event_counts.csv'),
    windows=None,
)
display(SETTINGS)

## Workflow implementation

The next cell defines functions only. It does not execute the workflow or write outputs.

In [ ]:
from __future__ import annotations

from pathlib import Path

from typing import Optional

import pandas as pd

from obspy import UTCDateTime

from obspy.core.event import Event

import pickle

from pinatubo_steps.shared import (get_comment_value)

COVER_WINDOWS = [
    ("1991-05-08", "1991-06-09"),
    ("1991-06-30", "1991-08-18"),
]

EXPECTED_LABELS = [
    "W_P_H",
    "W_P",
    "W_H",
    "W_ONLY",
    "P_H",
    "P_ONLY",
    "H_ONLY",
]

def _parse_utcdate(s: str) -> UTCDateTime:
    """Convert YYYY-MM-DD → UTCDateTime at start of day."""
    return UTCDateTime(f"{s}T00:00:00Z")

def _in_windows(t: UTCDateTime, windows) -> bool:
    """Check whether a UTCDateTime falls inside any coverage window."""
    for a, b in windows:
        start = _parse_utcdate(a)
        end = _parse_utcdate(b) + 24 * 3600 - 1e-6  # inclusive
        if start <= t <= end:
            return True
    return False

def get_event_class(event: Event) -> str:
    """
    Read event_class from Step 50.
    Falls back to UNKNOWN if missing.
    """
    val = get_comment_value(event, "event_class:")
    if val is None:
        return "UNKNOWN"
    return val.strip().upper()

def infer_event_time(event: Event) -> Optional[UTCDateTime]:
    """
    Choose a single representative time for binning:
      1) preferred origin time
      2) first origin time
      3) waveform_starttime comment
      4) earliest pick time
    """
    o = event.preferred_origin()
    if o and o.time:
        return o.time

    if event.origins:
        o0 = event.origins[0]
        if o0 and o0.time:
            return o0.time

    wst = get_comment_value(event, "waveform_starttime:")
    if wst:
        try:
            return UTCDateTime(wst)
        except Exception:
            pass

    if event.picks:
        times = [p.time for p in event.picks if p.time]
        if times:
            return min(times)

    return None

def run(
    *,
    catalog_pkl,
    out_daily_csv,
    windows=None
):
    """Run this Pinatubo step with explicit notebook arguments. Return tables and results for inspection."""
    from types import SimpleNamespace
    args = SimpleNamespace(
        catalog_pkl=catalog_pkl,
        out_daily_csv=out_daily_csv,
        windows=windows,
    )

    # Determine coverage windows
    if args.windows is None:
        windows = COVER_WINDOWS
    else:
        if len(args.windows) % 2 != 0:
            raise SystemExit("--windows must be start/end date pairs")
        windows = [
            (args.windows[i], args.windows[i + 1])
            for i in range(0, len(args.windows), 2)
        ]

    with open(args.catalog_pkl, "rb") as f:
        catalog = pickle.load(f)

    rows = []
    skipped_no_time = 0
    skipped_unknown = 0

    # ------------------------------------------------------------------
    # Iterate events
    # ------------------------------------------------------------------

    for ev in catalog:
        t = infer_event_time(ev)
        if t is None:
            skipped_no_time += 1
            continue

        if not _in_windows(t, windows):
            continue

        label = get_event_class(ev)
        if label not in EXPECTED_LABELS:
            skipped_unknown += 1
            continue

        day = pd.Timestamp(t.datetime).normalize()

        rows.append(
            {
                "date": day,
                "event_time_utc": t.datetime,
                "event_class": label,
            }
        )

    df = pd.DataFrame(rows)
    if df.empty:
        print("\nNo events found within the specified windows.")
        print("Skipped (no inferable time):", skipped_no_time)
        return

    # ------------------------------------------------------------------
    # Daily pivot table
    # ------------------------------------------------------------------

    daily = (
        df.pivot_table(
            index="date",
            columns="event_class",
            values="event_time_utc",
            aggfunc="count",
            fill_value=0,
        )
        .sort_index()
        .reset_index()
    )

    # Ensure consistent columns
    for col in EXPECTED_LABELS:
        if col not in daily.columns:
            daily[col] = 0

    daily["Total"] = daily[EXPECTED_LABELS].sum(axis=1)

    # ------------------------------------------------------------------
    # Output
    # ------------------------------------------------------------------

    out_csv = Path(args.out_daily_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    daily.to_csv(out_csv, index=False, date_format="%Y-%m-%d")

    # Overall composition
    comp = df["event_class"].value_counts().to_dict()
    total = len(df)

    print("\nEVENT-NUMBER STATS (restricted to coverage windows)")
    print("--------------------------------------------------")
    print("Windows (inclusive):")
    for a, b in windows:
        print(f"  - {a} .. {b}")
    print(f"\nTotal events in windows: {total}")
    for k in EXPECTED_LABELS:
        print(f"{k:22s}: {comp.get(k, 0)}")

    print(f"\nDaily CSV written: {out_csv}")
    if skipped_no_time:
        print(f"Skipped events with no inferable time: {skipped_no_time}")
    if skipped_unknown:
        print(f"Skipped events with unknown event_class: {skipped_unknown}")
    return {k: v for k, v in locals().items() if (k.startswith("df") or k.startswith("catalog") or k.endswith("_rows") or k in ("events", "annotations", "errors", "stats", "diag", "merged", "suppressed", "matches", "rows", "unparsed"))}

## Run and inspect

Run this cell only after the configuration and all definitions. It writes the configured outputs; inspect the summary and exceptions before proceeding.

In [ ]:
result = run(**SETTINGS)

In [ ]:
# Preview returned tables and summary objects without reloading output files.
for name, value in result.items():
    print(name)
    display(value.head() if hasattr(value, 'head') else
            value[:5] if isinstance(value, list) else value)